# 05 ClinVar pathogenic variant filtering

This notebook identifies ClinVar pathogenic or likely pathogenic variants and reviews conflicting ClinVar classifications using AutoGVP.

It saves the filtered variant tables, summary statistics, and figures.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

#project_dir = Path(r"C:\Users\katya\Box\KD_SUIP_noncoding_project")
#input_dir = project_dir / "sequencing_and_variantcalling_overview/on_off_target_qc"
#output_dir = project_dir / "pathogenic_filtering/clinvar"

notebook_dir = Path("/home/donetski/Notebooks")
input_dir = notebook_dir / "OutputFiles" / "04_qc_checking_on_target"
#output_dir = notebook_dir / "OutputFiles" / "05_clinvar_filtering"
output_dir = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/FinalPVs")

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
output_prefix = "05"
runs_to_process = "Run2"  # "all", "Run2", or ["Run2", "Run4"]

all_runs = ["Run1", "Run2", "Run3", "Run4"]
runs = all_runs if isinstance(runs_to_process, str) and runs_to_process.lower() == "all" else ([runs_to_process] if isinstance(runs_to_process, str) else runs_to_process)

print("Runs to process:", runs)

## Define the pathogenicity classifications

Variants are handled using three rules:

1. Clear ClinVar `Pathogenic` or `Likely_pathogenic` classifications are kept.
2. Conflicting ClinVar classifications are kept only when AutoGVP classifies the variant as pathogenic or likely pathogenic.
3. `Likely_pathogenic&protective` variants are saved separately for manual review.

In [ ]:
clinvar_keep = ["Pathogenic", "Likely_pathogenic", "Pathogenic/Likely_pathogenic"]

clinvar_conflicting = [
    "Conflicting_classifications_of_pathogenicity",
    "Conflicting_classifications_of_pathogenicity&risk_factor",
]

clinvar_review = ["Likely_pathogenic&protective"]

autogvp_keep = ["Pathogenic", "Likely_pathogenic", "Pathogenic/Likely_pathogenic"]

required_cols = [
    "Sample.ID", "Chr", "Start", "REF", "ALT", "Gene",
    "ClinVar.SIG", "AutoGVP", "matched_bed_region",
]

## Process the selected runs

For each run, the notebook:

- loads the full per-variant table produced by notebook 04;
- applies the ClinVar and AutoGVP filtering rules;
- records why each row was kept, excluded, or sent for manual review;
- creates gene, sample, carrier, and BED-region summaries;
- saves separate outputs and figures for that run.

Runs are processed one at a time to reduce memory usage.

In [ ]:
def process_run(run):
    #input_file = input_dir / f"04_{run}_per_variant_target_status_FULL.csv"
    #FINAL INPUT FILE
    input_file = Path("/project/knathans_shared/donetski/Notebooks/OutputFiles/FinalPVs/unique_variants_target_genes_sample_count_lt50_VAF_ge1pct_AltDepth_ge5.csv")
    
    df = pd.read_csv(input_file, low_memory=False)

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{run} is missing columns: {missing_cols}")

    clear_pathogenic = df["ClinVar.SIG"].isin(clinvar_keep)
    conflicting_pathogenic = df["ClinVar.SIG"].isin(clinvar_conflicting) & df["AutoGVP"].isin(autogvp_keep)
    manual_review = df["ClinVar.SIG"].isin(clinvar_review)

    df["pathogenic_filter_reason"] = "not_kept"
    df.loc[clear_pathogenic, "pathogenic_filter_reason"] = "kept_clear_clinvar_pathogenic"
    df.loc[conflicting_pathogenic, "pathogenic_filter_reason"] = "kept_conflicting_clinvar_but_autogvp_pathogenic"
    df.loc[manual_review, "pathogenic_filter_reason"] = "manual_review_likely_pathogenic_protective"

    pathogenic = df[df["pathogenic_filter_reason"].str.startswith("kept")].copy()
    manual = df[df["pathogenic_filter_reason"].str.startswith("manual")].copy()

    pathogenic["variant_id"] = (
        pathogenic["Chr"].astype(str) + ":" +
        pathogenic["Start"].astype(str) + ":" +
        pathogenic["REF"].astype(str) + ">" +
        pathogenic["ALT"].astype(str)
    )

    gene_summary = (
        pathogenic.groupby("Gene")
        .agg(
            n_pathogenic_variants=("variant_id", "nunique"),
            n_rows=("variant_id", "size"),
            n_samples=("Sample.ID", "nunique"),
        )
        .reset_index()
        .sort_values("n_pathogenic_variants", ascending=False)
    )

    sample_summary = (
        pathogenic.groupby("Sample.ID")
        .agg(
            n_pathogenic_variants=("variant_id", "nunique"),
            n_genes=("Gene", "nunique"),
            genes=("Gene", lambda x: ", ".join(sorted(set(x.dropna().astype(str))))),
        )
        .reset_index()
        .sort_values("n_pathogenic_variants", ascending=False)
    )

    gene_carrier_summary = (
        pathogenic.drop_duplicates(["Sample.ID", "Gene"])
        .groupby("Gene", as_index=False)
        .agg(n_carrier_samples=("Sample.ID", "nunique"))
        .sort_values("n_carrier_samples", ascending=False)
    )

    gene_bed_check = pathogenic[
        ["Sample.ID", "variant_id", "Gene", "matched_bed_region"]
    ].drop_duplicates()

    gene_bed_check["gene_matches_bed"] = gene_bed_check.apply(
        lambda row: str(row["Gene"]) in str(row["matched_bed_region"]), axis=1
    )
    gene_bed_mismatches = gene_bed_check[~gene_bed_check["gene_matches_bed"]].copy()

    print(f"{run}: {len(df):,} rows | {len(pathogenic):,} pathogenic | {len(manual):,} manual review")

    return {
        "pathogenic": pathogenic,
        "manual": manual,
        "gene_summary": gene_summary,
        "sample_summary": sample_summary,
        "gene_carrier_summary": gene_carrier_summary,
        "gene_bed_check": gene_bed_check,
        "gene_bed_mismatches": gene_bed_mismatches,
    }

## Save the filtered tables and summaries

Each run receives:

- a full pathogenic-variant workbook;
- a separate manual-review workbook;
- a summary workbook containing the full pathogenic table and all summary tables.

All files created by this notebook begin with the `05` prefix.

In [ ]:
def save_run_outputs(run, results):
    pathogenic_file = output_dir / f"{output_prefix}_{run}_pathogenic_variants_filtered.xlsx"
    manual_file = output_dir / f"{output_prefix}_{run}_manual_review_variants_filtered.xlsx"
    summary_file = output_dir / f"{output_prefix}_{run}_pathogenic_variant_summary_stats.xlsx"

    results["pathogenic"].to_excel(pathogenic_file, index=False)
    results["manual"].to_excel(manual_file, index=False)

    with pd.ExcelWriter(summary_file) as writer:
        results["pathogenic"].to_excel(writer, sheet_name="pathogenic_variants_full", index=False)
        results["gene_summary"].to_excel(writer, sheet_name="gene_summary", index=False)
        results["sample_summary"].to_excel(writer, sheet_name="sample_summary", index=False)
        results["gene_carrier_summary"].to_excel(writer, sheet_name="gene_carrier_summary", index=False)
        results["gene_bed_check"].to_excel(writer, sheet_name="gene_bed_check", index=False)
        results["gene_bed_mismatches"].to_excel(writer, sheet_name="gene_bed_mismatches", index=False)

    print(f"Saved {run} spreadsheets")

## Plot pathogenic variants by gene and sample

The first figure shows the number of unique pathogenic variants per gene. The second shows the number of unique pathogenic variants per affected sample.

Each figure is displayed in the notebook and saved with the notebook-05 prefix.

In [ ]:
def plot_run_outputs(run, results):
    gene_summary = results["gene_summary"].sort_values("n_pathogenic_variants")
    sample_summary = results["sample_summary"]

    plt.figure(figsize=(8, 6))
    plt.barh(gene_summary["Gene"], gene_summary["n_pathogenic_variants"])
    plt.xlabel("Unique pathogenic variants")
    plt.ylabel("Gene")
    plt.title(f"Pathogenic variants per gene - {run}")
    plt.tight_layout()
    plt.savefig(output_dir / f"{output_prefix}_{run}_pathogenic_variants_per_gene.png", dpi=300)
    plt.show()
    plt.close()

    plt.figure(figsize=(12, 5))
    plt.bar(sample_summary["Sample.ID"].astype(str), sample_summary["n_pathogenic_variants"])
    plt.xlabel("Sample")
    plt.ylabel("Unique pathogenic variants")
    plt.title(f"Pathogenic variants per sample - {run}")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.savefig(output_dir / f"{output_prefix}_{run}_pathogenic_variants_per_sample.png", dpi=300)
    plt.savefig
    plt.show()
    plt.close()

## Run the filtering workflow

Process each selected run independently. Running all runs together still creates separate files and figures for each run.

In [ ]:
for run in runs:
    results = process_run(run)
    save_run_outputs(run, results)
    plot_run_outputs(run, results)